<a href="https://colab.research.google.com/github/DanieleBaiocco/nlp_first_assignment/blob/main/Assignment1_rob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import necessary libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from typing import List, Dict
import pandas as pd
from tqdm import tqdm
from matplotlib import pyplot as plt
import nltk

import seaborn as sns

from gensim.models import KeyedVectors

from keras.utils import pad_sequences
from keras.utils.np_utils import to_categorical
from keras.models import Sequential
from keras.layers import Embedding
from keras.layers import Dense, Input
from keras.layers import TimeDistributed
from keras.layers import LSTM, GRU, Bidirectional, SimpleRNN, RNN
from keras.models import Model
from keras.preprocessing.text import Tokenizer

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

In [2]:
import os
from urllib import request
from zipfile import ZipFile
import numpy as np
import pandas as pd

print(f"Current work directory: {os.getcwd()}")
dataset_folder = os.path.join(os.getcwd(), "Datasets")

if not os.path.exists(dataset_folder):
    os.makedirs(dataset_folder)

url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/dependency_treebank.zip"
dataset_path = os.path.join(dataset_folder, "Corpora.zip")
print(dataset_path)

def download_dataset(download_path: str, url: str):
    if not os.path.exists(download_path):
        print("Downloading dataset...")
        request.urlretrieve(url, download_path)
        print("Download complete!")

def extract_dataset(download_path: str, extract_path: str):
    print("Extracting dataset... (it may take a while...)")
    with ZipFile(download_path, 'r') as zObject:
        zObject.extractall(
		path=extract_path)
    print("Extraction completed!")

download_dataset(dataset_path, url)
extract_dataset(dataset_path, dataset_folder)

Current work directory: /content
/content/Datasets/Corpora.zip
Extracting dataset... (it may take a while...)
Extraction completed!


In [3]:
def split_sentences(end) -> pd.DataFrame:
    dataset_folder = os.path.join(os.getcwd(), "Datasets", "dependency_treebank")
    list_dir = os.listdir(dataset_folder) 
    dataframe_rows = []
    count = 0 
    for dir in range(end):
      file_path = os.path.join(dataset_folder, list_dir[dir])   
      if dir == 0 or dir==1:
        print(file_path)               
      try:
        if os.path.isfile(file_path):
          with open(file_path, mode = 'r', encoding = 'utf-8') as text_file:
            collected_lines = []
            for line in text_file:
              if len(line) == 1:
                count = count + 1
      except Exception as e:
            print('Failed to process %s. Reason: %s' % (file_path, e))
            sys.exit(0)
    print(count)
    return count

In [4]:
def read_sentence():
    dataset_folder = os.path.join(os.getcwd(), "Datasets", "dependency_treebank")
    list_dir = os.listdir(dataset_folder) 
    list_dir.sort()

    document = []
    documents = []
    sentence = []
    document_count = 0
    for dir in list_dir:
      file_path = os.path.join(dataset_folder, dir)               
      try:
          if os.path.isfile(file_path):
            with open(file_path, mode = 'r', encoding = 'utf-8') as text_file:
              document = []
              lines = text_file.readlines()
              for count, line in enumerate(lines):
                splitted_line = line.split()
                if len(splitted_line) == 3:
                  sentence.append((splitted_line[0], splitted_line[1]))
                if (len(splitted_line) != 3 or count + 1 >= len(lines)):
                  document.append(sentence)
                  sentence = []
              documents.append(document)
      except Exception as e:
            print('Failed to process %s. Reason: %s' % (file_path, e))
            sys.exit(0)
    return pd.DataFrame(documents)

documents = read_sentence()

In [21]:
documents.iloc[:3]

,0,1,2,3,4,5,6,7,8,9,...,175,176,177,178,179,180,181,182,183,184
0,"[(Pierre, NNP), (Vinken, NNP), (,, ,), (61, CD...","[(Mr., NNP), (Vinken, NNP), (is, VBZ), (chairm...",None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,"[(Rudolph, NNP), (Agnew, NNP), (,, ,), (55, CD...",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,"[(A, DT), (form, NN), (of, IN), (asbestos, NN)...","[(The, DT), (asbestos, NN), (fiber, NN), (,, ,...","[(Lorillard, NNP), (Inc., NNP), (,, ,), (the, ...","[(Although, IN), (preliminary, JJ), (findings,...","[(A, DT), (Lorillard, NNP), (spokewoman, NN), ...","[(We, PRP), ('re, VBP), (talking, VBG), (about...","[(There, EX), (is, VBZ), (no, DT), (asbestos, ...","[(Neither, DT), (Lorillard, NNP), (nor, CC), (...","[(``, ``), (We, PRP), (have, VBP), (no, DT), (...","[(Dr., NNP), (Talcott, NNP), (led, VBD), (a, D...",...,None,None,None,None,None,None,None,None,None,None


In [6]:
train_set = documents[:100]
val_set = documents[100:150]
test_set = documents[150:]

In [7]:
train_set.iloc[0][1]

[('Mr.', 'NNP'),
 ('Vinken', 'NNP'),
 ('is', 'VBZ'),
 ('chairman', 'NN'),
 ('of', 'IN'),
 ('Elsevier', 'NNP'),
 ('N.V.', 'NNP'),
 (',', ','),
 ('the', 'DT'),
 ('Dutch', 'NNP'),
 ('publishing', 'VBG'),
 ('group', 'NN'),
 ('.', '.')]

Splitting the sets in X(words) and y(labels)

In [8]:
def split_word_tag(dataframe):
  X = []
  y = []
  tempX = []
  tempy = []
  for i in range(dataframe.shape[0]):
      for col in dataframe.iloc[i]:
        if col:
          for couple in col:
            tempX.append(couple[0])
            tempy.append(couple[1])
          X.append(tempX)
          y.append(tempy)
          tempX =[]
          tempy = []
  return X, y

In [9]:
X_train, y_train = split_word_tag(train_set)
X_val, y_val = split_word_tag(val_set)
X_test, y_test = split_word_tag(test_set)

In [10]:
!pip install gensim

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [11]:
import gensim
import gensim.downloader as gloader

def load_embedding_model(model_type: str,
                         embedding_dimension: int = 50) -> gensim.models.keyedvectors.KeyedVectors:
    """
    Loads a pre-trained word embedding model via gensim library.

    :param model_type: name of the word embedding model to load.
    :param embedding_dimension: size of the embedding space to consider

    :return
        - pre-trained word embedding model (gensim KeyedVectors object)
    """
    download_path = ""
    if model_type.strip().lower() == 'glove':
        download_path = "glove-wiki-gigaword-{}".format(embedding_dimension)
    elif model_type.strip().lower() == 'fasttext':
        download_path = "fasttext-wiki-news-subwords-300"
    else:
        raise AttributeError("Unsupported embedding model type! Available ones: word2vec, glove, fasttext")
        
    try:
        emb_model = gloader.load(download_path)
    except ValueError as e:
        print("Invalid embedding model name! Check the embedding dimension:")
        print("Word2Vec: 300")
        print("Glove: 50, 100, 200, 300")
        print('FastText: 300')
        raise e

    return emb_model

Creating GloVe


In [30]:
embedding_model = load_embedding_model(model_type="glove",
                                       embedding_dimension=100)

In [13]:
def check_OOV_terms(embedding_model: gensim.models.keyedvectors.KeyedVectors,
                    dataset: List[str]):
    """
    Checks differences between pre-trained embedding model vocabulary
    and dataset specific vocabulary in order to highlight out-of-vocabulary terms.

    :param embedding_model: pre-trained word embedding model (gensim wrapper)
    :param word_listing: dataset specific vocabulary (list)

    :return
        - list of OOV terms
    """
    embedding_vocabulary = set(embedding_model.vocab.keys())
    word_listing = {word for sentence in dataset for word in sentence} #flat list
    oov = set(word_listing).difference(embedding_vocabulary)

    oov_percentage = float(len(list(oov))) * 100 / len(word_listing)
    print(f"Total OOV terms: {len(list(oov))} ({oov_percentage:.2f}%)")
    return list(oov)

Function that adds out of vocabulary words into the original GloVe vocabulary

In [28]:
def add_OOV_terms(vocabulary, oov, embedding_dim):
  for word in oov:
    vocabulary[word] = np.random.uniform(-1, 1, size=embedding_dim)
  print(f"Generated embeddings for {len(oov)} OOV words.")
  return vocabulary

In [31]:
v1 = embedding_model.vocab
oov_terms_train = check_OOV_terms(embedding_model, X_train)
v2 = add_OOV_terms(v1, oov_terms_train, embedding_dim=100)

Total OOV terms: 2346 (29.29%)
Generated embeddings for 2346 OOV words.


In [32]:
oov_terms_val = check_OOV_terms(embedding_model, X_val)
v3 = add_OOV_terms(v2, oov_terms_val, embedding_dim=100)

Total OOV terms: 944 (16.02%)
Generated embeddings for 944 OOV words.


In [33]:
oov_terms_test = check_OOV_terms(embedding_model, X_test)
v4 = add_OOV_terms(v3, oov_terms_test, embedding_dim=100)

Total OOV terms: 455 (12.56%)
Generated embeddings for 455 OOV words.


il valore 3745 è corretto

In [41]:
print(len(v4))

403745
